# Day 2: train baseline and redacted-data models

This notebook uses GPT-2, matching the reference project, to train two small experimental versions: one on the original fictional conversations and one on the same conversations with private values replaced by `[REDACTED]`. Training both with the same model, seed, and settings gives us a fairer comparison.

This notebook logs assistant-answer-only training and validation loss to `results/training_metrics.csv`, appending one row per model per epoch with a shared run ID. It masks resident messages from the loss while retaining them as context. The old notebook outputs below were produced before this fix and are not comparable; rerun the training cell for valid metrics. The training code uses the 200 validation conversations and never trains on validation or test records.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "data" / "synthetic" / "training_conversations.jsonl").exists(), (
    "Run notebook 01 or generate the Day 1 dataset first."
)
print(f"Project root: {PROJECT_ROOT}")
print("Install dependencies in the next cell if needed.")

Project root: /Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage
Install dependencies in the next cell if needed.


In [ ]:
# Uncomment this in Colab or a fresh environment to install project dependencies.
# %pip install -r requirements.txt

## Configure a run

Set `TRAIN_EXAMPLES` to an integer for a reproducible random subset (for example 250), or `None` to use all 1,000 records. Both model conditions use the same subset. Each run appends metrics; checkpoints use a separate output directory so older runs are preserved. GPT-2 weights must be downloaded once.

In [ ]:
import subprocess
from datetime import datetime

TRAIN_EXAMPLES = 250  # Set to None for all 1,000 training records.
EPOCHS = 1
RUN_LABEL = f"n{TRAIN_EXAMPLES or 'all'}-e{EPOCHS}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
command = [
    sys.executable,
    str(PROJECT_ROOT / "src" / "train_models.py"),
    "--data-dir", str(PROJECT_ROOT / "data" / "synthetic"),
    "--model-name", "gpt2",
    "--output-dir", str(PROJECT_ROOT / "models" / "day2-answer-only" / RUN_LABEL),
    "--metrics-csv", str(PROJECT_ROOT / "results" / "training_metrics.csv"),
    "--seed", "42",
    "--epochs", str(EPOCHS),
    "--batch-size", "1",
    "--gradient-accumulation-steps", "8",
    "--max-length", "128",
]
if TRAIN_EXAMPLES is not None:
    command.extend(["--max-train-examples", str(TRAIN_EXAMPLES)])
print(f"Training both conditions with {TRAIN_EXAMPLES or 1000} examples for {EPOCHS} epoch(s).")
subprocess.run(command, cwd=PROJECT_ROOT, check=True)


Training baseline with 100 conversations.
Device: mps; base model: gpt2


W1105 08:33:40.019000 95328 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5618.78it/s]
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


baseline: epoch 1/1, mean training loss 2.0613


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.88s/it]


Saved baseline model and run configuration to /Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/models/day2-smoke/baseline

Training redacted with 100 conversations.
Device: mps; base model: gpt2


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5099.58it/s]


redacted: epoch 1/1, mean training loss 1.6605


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.05s/it]


Saved redacted model and run configuration to /Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/models/day2-smoke/redacted


CompletedProcess(args=['/opt/miniconda3/bin/python', '/Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/src/train_models.py', '--data-dir', '/Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/data/synthetic', '--model-name', 'gpt2', '--output-dir', '/Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/models/day2-smoke', '--seed', '42', '--epochs', '1', '--batch-size', '1', '--gradient-accumulation-steps', '8', '--max-length', '128', '--max-train-examples', '100'], returncode=0)

## Suggested training comparisons

First run 250 examples for one epoch as a pipeline/learning diagnostic. Then set `TRAIN_EXAMPLES = None`, keeping one epoch, and run again. Finally, run all examples for two epochs to check whether an extra pass helps or overfits. Every condition and epoch gets an appended CSV row; each run has a different ID. Keep seed and other settings fixed while comparing sample size or epochs.

These are three dataset/epoch configurations, each training both baseline and redacted models. They are not three independent random seeds. Do not tune against the test split.